# 03 - 协作式多智能体教程 (Collaborative Multi-Agent)

## 学习目标

1. **协作模式** - 顺序、并行、轮询
2. **团队配置** - 创建和管理智能体团队
3. **共识构建** - 多智能体达成共识
4. **投票系统** - 民主决策机制

---

## 协作 vs 辩论

| 特性 | 协作式 | 辩论式 |
|------|--------|--------|
| 目标 | 共同完成任务 | 对抗性推理 |
| 关系 | 合作 | 竞争 |
| 输出 | 综合结果 | 胜负裁决 |

In [ ]:
# 环境设置
import sys
import asyncio
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from agent_base import AgentConfig, MockLLM, create_agent
from collaborative_agents import (
    CollaborationMode, TeamConfig,
    CollaborativeTeam, ConsensusBuilder, VotingSystem
)

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(coro)

print('=' * 60)
print('多智能体系统 - 协作教程')
print('=' * 60)

---

## 第一部分：协作模式

In [ ]:
# 1.1 查看协作模式
print('协作模式:')
modes = {
    'sequential': '顺序执行',
    'parallel': '并行执行',
    'round_robin': '轮询执行',
}
for mode in CollaborationMode:
    print(f'  {mode.name}: {modes.get(mode.value, mode.value)}')

---

## 第二部分：创建协作团队

In [ ]:
# 2.1 创建团队成员
print('=' * 60)
print('创建团队')
print('=' * 60)

researcher = create_agent('研究员', llm=MockLLM(responses=['研究发现：MAS起源于分布式AI。']))
writer = create_agent('撰稿人', llm=MockLLM(responses=['基于研究撰写：MAS是分布式AI分支。']))
editor = create_agent('编辑', llm=MockLLM(responses=['编辑后：MAS是分布式AI的重要分支。']))

print(f'成员: {researcher.name}, {writer.name}, {editor.name}')

In [ ]:
# 2.2 组建团队
config = TeamConfig(name='内容团队', mode=CollaborationMode.SEQUENTIAL)
team = CollaborativeTeam(config)
team.add_member(researcher, '信息收集')
team.add_member(writer, '内容撰写')
team.add_member(editor, '文章润色')

print(f'团队: {team.config.name}, 成员数: {len(team._members)}')

---

## 第三部分：运行协作任务

In [ ]:
# 3.1 执行顺序协作
async def run_collab():
    task = '撰写多智能体系统简介'
    print(f'任务: {task}')
    result = await team.collaborate(task)
    print(f'结果: {result}')
    return result

run_async(run_collab())

In [ ]:
# 3.2 查看贡献记录
contributions = team.get_contributions()
print(f'\n贡献记录 ({len(contributions)} 条):')
for c in contributions:
    print(f'  [{c.agent_name}] {c.content[:40]}...')

---

## 第四部分：投票系统

In [ ]:
# 4.1 创建投票系统
voter1 = create_agent('专家1', llm=MockLLM(responses=['1']))
voter2 = create_agent('专家2', llm=MockLLM(responses=['2']))
voter3 = create_agent('专家3', llm=MockLLM(responses=['1']))

voting = VotingSystem([voter1, voter2, voter3])
print(f'投票者: {[a.name for a in voting.agents]}')

In [ ]:
# 4.2 进行投票
async def run_vote():
    question = '优先开发什么功能？'
    options = ['性能优化', '新功能', 'Bug修复']
    result = await voting.vote(question, options)
    print(f'获胜: {result["winner"]}')
    print(f'统计: {result["tally"]}')

run_async(run_vote())

---

## 第五部分：共识构建

In [ ]:
# 5.1 创建共识构建器
e1 = create_agent('保守派', llm=MockLLM(responses=['优先稳定性', '同意平衡方案']))
e2 = create_agent('激进派', llm=MockLLM(responses=['大胆创新', '同意平衡方案']))
e3 = create_agent('中立派', llm=MockLLM(responses=['平衡方案', '达成共识']))

consensus = ConsensusBuilder(agents=[e1, e2, e3], threshold=0.7, max_iterations=2)
print(f'参与者: {[a.name for a in consensus.agents]}')

In [ ]:
# 5.2 构建共识
async def build():
    result = await consensus.build_consensus('技术战略方向？')
    print(f'共识: {result.final_answer}')
    print(f'一致性: {result.agreement_score:.2f}')

run_async(build())

---

## 总结

本教程涵盖：
1. **CollaborationMode**: 顺序、并行、轮询
2. **CollaborativeTeam**: 协作团队
3. **VotingSystem**: 投票决策
4. **ConsensusBuilder**: 共识构建